# Project findings

This notebook is mainly a project summary. It checks the main derived files and records the current project state.

The actual baseline model run and the main classification metrics are in `06_model_roadwork_weather.ipynb`.


In [1]:
from pathlib import Path

import pandas as pd

DERIVED_DIR = Path("data/derived")
RAW_ROADWORK_DIR = Path("data/raw/roadwork")
RAW_PLOW_DIR = Path("data/raw/plow")


In [2]:
street_weather = pd.read_csv(DERIVED_DIR / "street_weather.csv")
resurfacing = pd.read_csv(DERIVED_DIR / "resurfacing_agg_full.csv")
traffic = pd.read_csv(DERIVED_DIR / "traffic_agg.csv")
plow = pd.read_csv(DERIVED_DIR / "plow_df.csv")
snowfall = pd.read_csv(DERIVED_DIR / "snowfall_df.csv")
lagged = pd.read_csv(DERIVED_DIR / "street_weather_lagged_model.csv")


In [3]:
inventory = pd.DataFrame([
    {"dataset": "street_weather", "rows": len(street_weather), "unique_streets": street_weather["normalized_street_name_season"].str.rsplit("_", n=1).str[0].nunique()},
    {"dataset": "resurfacing_agg_full", "rows": len(resurfacing), "unique_streets": resurfacing["normalized_street_name"].nunique()},
    {"dataset": "traffic_agg", "rows": len(traffic), "unique_streets": traffic["normalized_street_name"].nunique()},
    {"dataset": "plow_df", "rows": len(plow), "unique_streets": plow["normalized_street_name_season"].str.rsplit("_", n=1).str[0].nunique()},
    {"dataset": "snowfall_df", "rows": len(snowfall), "unique_streets": snowfall["normalized_street_name"].nunique()},
    {"dataset": "street_weather_lagged_model", "rows": len(lagged), "unique_streets": lagged["normalized_street_name"].nunique()},
])
inventory


,dataset,rows,unique_streets
0,street_weather,119990,11999
1,resurfacing_agg_full,20671,6985
2,traffic_agg,818,818
3,plow_df,69078,11399
4,snowfall_df,60000,12000
5,street_weather_lagged_model,276000,12000


In [4]:
snapshot_findings = {
    "lagged_year_min": int(lagged["year"].min()),
    "lagged_year_max": int(lagged["year"].max()),
    "roadwork_positive_rate": float(lagged["roadwork_done"].mean()),
    "nonzero_roadwork_seasons": int((resurfacing["roadwork_factor"].fillna(0) > 0).sum()),
    "mean_traffic_volume": float(traffic["Vol"].mean()),
    "missing_raw_roadwork": not (RAW_ROADWORK_DIR / "dot_inhouse_resurfacing.csv").exists(),
    "missing_raw_plow_dir_contents": len(list(RAW_PLOW_DIR.glob("plow_*.csv"))) == 0,
}
snapshot_findings


{'lagged_year_min': 2003,
 'lagged_year_max': 2025,
 'roadwork_positive_rate': 0.06987318840579711,
 'nonzero_roadwork_seasons': 19986,
 'mean_traffic_volume': 120.13611137497212,
 'missing_raw_roadwork': False,
 'missing_raw_plow_dir_contents': False}

Current state: this notebook is mainly a project summary and dataset check. If you want the actual model fit, ROC-AUC, precision, recall, and feature importances, look at `06_model_roadwork_weather.ipynb`.


Quick plots for scale:

The first plot just shows how large each main derived table is. The second shows the fraction of positive roadwork rows by year in the final modeling table.

In [ ]:
import matplotlib.pyplot as plt

inventory_plot = inventory.copy()
yearly_rate = lagged.groupby("year")["roadwork_done"].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(inventory_plot["dataset"], inventory_plot["rows"], color="#4C78A8")
axes[0].set_title("Rows in main derived tables")
axes[0].set_ylabel("Rows")
axes[0].tick_params(axis="x", rotation=45)

axes[1].plot(yearly_rate["year"], yearly_rate["roadwork_done"], marker="o", color="#E45756")
axes[1].set_title("Positive roadwork rate by year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Share of positive rows")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()